In [ ]:
import os, sys, re, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from collections import defaultdict

from MAE_model_downstream import PedSleepMAE
from utils.misc import setup_seed
from dataloader import HDF5Dataset

warnings.filterwarnings("ignore")

# config
search_label = "apnea_label"
data_dir = "./hdf5_data"
checkpoint_file = "./checkpoints/mae_checkpoint.pt"

patch_size = 8
mask_ratio = 15
emb_dim = 64
num_head = 4
num_layer = 3
patient_ids = ["pid1"]
study_ids = ["sid1"]
seed = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
num_patches = int(3840 / patch_size)

setup_seed(seed)

# helpers
def extract_sample_id(fname):
    m = re.search(r"_sample_(\d+)\.hdf5$", fname)
    return int(m.group(1)) if m else float("inf")

# gather files
files = [
    os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".hdf5")
]
files = [
    f for f in files
    if f.split("/")[-1].split("_")[0] in patient_ids
    and f.split("/")[-1].split("_")[1] in study_ids
]
files = sorted(files, key=extract_sample_id)

if len(files) == 0:
    print("no matching patient/session"); sys.exit(1)

print(f"total sorted files: {len(files)}")
print("first few:", files[:5])

# load model
model = PedSleepMAE(
    batch_size=len(files),
    patch_size=patch_size,
    mask_ratio=mask_ratio,
    emb_dim=emb_dim,
    num_head=num_head,
    num_layer=num_layer,
).to(device)

ckpt = torch.load(checkpoint_file, map_location=device, weights_only=True)
model.load_state_dict(ckpt["state_dict"])

print("device:", device)

# dataloader
dataset = HDF5Dataset(files, search_label)
loader = DataLoader(dataset, batch_size=100, shuffle=False)

# run
pool = nn.AdaptiveMaxPool1d(1)
emb_list = []

for b, (signal, _, _) in enumerate(loader):
    print("batch", b)
    with torch.no_grad():
        sig = signal.squeeze().float().to(device)
        n, _, _ = sig.shape
        feats, _ = model.encoder(sig)
        feats = feats[:, :, 1:, :]
        feats = feats.reshape(-1, num_patches, emb_dim)
        pooled = pool(feats).reshape(n, -1)
        emb_list.extend(pooled.cpu().numpy())

emb_array = np.vstack(emb_list)

prefix = f"{'_'.join(patient_ids)}_{'_'.join(study_ids)}"
os.makedirs("./output_embeddings_sorted", exist_ok=True)
np.save(f"./output_embeddings_sorted/{prefix}_embeddings.npy", emb_array)

print("saved:", f"./output_embeddings_sorted/{prefix}_embeddings.npy", emb_array.shape)


In [ ]:
import numpy as np
import phate
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

embeddings = np.load("./output_embeddings_sorted/example_embeddings.npy")
time_idx = np.arange(len(embeddings))

def plot_2D_methods(embeddings, time_idx):
    print("running PCA, t-SNE, PHATE...")

    pca = PCA(n_components=2).fit_transform(embeddings)
    tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42).fit_transform(embeddings)
    phate_op = phate.PHATE(n_components=2, knn=5, t=12, random_state=42).fit_transform(embeddings)

    cmap = plt.get_cmap("turbo")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    sc1 = axes[0].scatter(pca[:,0], pca[:,1], c=time_idx, cmap=cmap, s=10, alpha=0.8)
    axes[0].set_title("PCA"); fig.colorbar(sc1, ax=axes[0], label="time")

    sc2 = axes[1].scatter(tsne[:,0], tsne[:,1], c=time_idx, cmap=cmap, s=10, alpha=0.8)
    axes[1].set_title("t-SNE"); fig.colorbar(sc2, ax=axes[1], label="time")

    sc3 = axes[2].scatter(phate_op[:,0], phate_op[:,1], c=time_idx, cmap=cmap, s=10, alpha=0.8)
    axes[2].set_title("PHATE"); fig.colorbar(sc3, ax=axes[2], label="time")

    plt.tight_layout()
    plt.show()

plot_2D_methods(embeddings, time_idx)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import phate
import matplotlib.pyplot as plt

def plot_3D_methods(embeddings, time_idx):
    print("running PCA, t-SNE, PHATE in 3D...")

    pca = PCA(n_components=3).fit_transform(embeddings)
    tsne = TSNE(n_components=3, perplexity=30, learning_rate=200, random_state=42).fit_transform(embeddings)
    phate_op = phate.PHATE(n_components=3, knn=5, t=12, random_state=42).fit_transform(embeddings)

    cmap = plt.get_cmap("turbo")
    fig = plt.figure(figsize=(18, 6))

    ax1 = fig.add_subplot(131, projection="3d")
    s1 = ax1.scatter(pca[:,0], pca[:,1], pca[:,2], c=time_idx, cmap=cmap, s=5, alpha=0.7)
    ax1.set_title("PCA 3D"); fig.colorbar(s1, ax=ax1, label="time")

    ax2 = fig.add_subplot(132, projection="3d")
    s2 = ax2.scatter(tsne[:,0], tsne[:,1], tsne[:,2], c=time_idx, cmap=cmap, s=5, alpha=0.7)
    ax2.set_title("t-SNE 3D"); fig.colorbar(s2, ax=ax2, label="time")

    ax3 = fig.add_subplot(133, projection="3d")
    s3 = ax3.scatter(phate_op[:,0], phate_op[:,1], phate_op[:,2], c=time_idx, cmap=cmap, s=5, alpha=0.7)
    ax3.set_title("PHATE 3D"); fig.colorbar(s3, ax=ax3, label="time")

    plt.tight_layout()
    plt.show()

plot_3D_methods(embeddings, time_idx)

In [ ]:
import numpy as np
import phate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

embeddings = np.load("./output_embeddings_sorted/example_embeddings.npy")
time_idx = np.arange(len(embeddings))

def group_time_idx(time_idx):
    tmin = time_idx * 0.5
    bins = []
    for t in tmin:
        if t < 60: bins.append("0-60")
        elif t < 120: bins.append("60-120")
        elif t < 180: bins.append("120-180")
        elif t < 240: bins.append("180-240")
        elif t < 300: bins.append("240-300")
        else: bins.append("300+")
    return bins

def plot_2D_methods(emb, time_idx):
    time_bins = group_time_idx(time_idx)
    labels = ["0-60","60-120","120-180","180-240","240-300","300+"]
    palette = dict(zip(labels, sns.color_palette("Spectral", len(labels))))
    colors = [palette[l] for l in time_bins]

    pca = PCA(n_components=2).fit_transform(emb)
    tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42).fit_transform(emb)
    phate_op = phate.PHATE(n_components=2, knn=5, t=20, n_pca=100, random_state=42).fit_transform(emb)

    fig, axes = plt.subplots(1, 3, figsize=(18,6))
    axes[0].scatter(pca[:,0], pca[:,1], c=colors, s=10, alpha=0.8); axes[0].set_title("PCA")
    axes[1].scatter(tsne[:,0], tsne[:,1], c=colors, s=10, alpha=0.8); axes[1].set_title("t-SNE")
    axes[2].scatter(phate_op[:,0], phate_op[:,1], c=colors, s=10, alpha=0.8); axes[2].set_title("PHATE")

    handles = [plt.Line2D([0],[0], marker="o", color="w", label=l, markersize=10, markerfacecolor=c)
               for l,c in palette.items()]
    fig.legend(handles=handles, loc="upper right", title="time (min)")
    plt.tight_layout()
    plt.show()

plot_2D_methods(embeddings, time_idx)